In [2]:
!pip install pycryptodome cryptography

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 25.2 MB/s eta 0:00:00


In [4]:
import socket
import ssl
import cryptography.x509 as x509
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, ec

def get_target_modulus(hostname):
    print(f"Analizando seguridad en: {hostname}...")
    context = ssl.create_default_context()
    try:
        with socket.create_connection((hostname, 443), timeout=10) as sock:
            with context.wrap_socket(sock, server_hostname=hostname) as ssock:
                cert_der = ssock.getpeercert(binary_form=True)
                cert = x509.load_der_x509_certificate(cert_der, default_backend())
                public_key = cert.public_key()

                if isinstance(public_key, rsa.RSAPublicKey):
                    n = public_key.public_numbers().n
                    bits = n.bit_length()
                    return n, bits
                elif isinstance(public_key, ec.EllipticCurvePublicKey):
                    print(f"⚠️ {hostname} ya se pasó a Curva Elíptica (ECC). No hay módulo N.")
                    return None, None
                else:
                    print(f"Tipo de clave desconocido en {hostname}")
                    return None, None
    except Exception as e:
        print(f"Error conectando a {hostname}: {e}")
        return None, None

# El objetivo de tu predicción
target = 'jpmorgan.com'
N_jpm, bit_length = get_target_modulus(target)

if N_jpm:
    print(f"\n✅ ¡OBJETIVO LOCALIZADO! RSA DETECTADO.")
    print(f"Módulo N de JP Morgan ({bit_length} bits):")
    print("-" * 60)
    print(N_jpm)
    print("-" * 60)
    print("\n[!] Listo para procesar en la TPU v5e...")

Analizando seguridad en: jpmorgan.com...

✅ ¡OBJETIVO LOCALIZADO! RSA DETECTADO.
Módulo N de JP Morgan (2048 bits):
------------------------------------------------------------
20401308423288094242766662999493793187525420878197109253818355727956826504712629708789530022692909325272817125236977390052166930825056004028872464770803083249959025095910645332450783024034058817080357808464693098920369306132436800716130122636415040330812678918561101207970271295199148388284056864933825212541980308771716287643934228961790479259363124623312430892043194378423348328847933501339676855519527154465017072352029567538488597745797965852775230430579578162516042451489532165421970081208347386436907444609093030460498848125912815856038414252622303104009405061242804767128891944731455892392535411542580604029973
------------------------------------------------------------

[!] Listo para procesar en la TPU v5e...
